# CS 6220 Homework 4


## Part 1 Frequent Itemsets


In [ ]:
from itertools import combinations

transactions = [
    ["Bread", "Butter", "Milk"],
    ["Bread", "Butter", "Eggs"],
    ["Milk", "Cereal"],
    ["Bread", "Eggs"],
    ["Milk", "Cereal", "Yogurt"],
    ["Diapers", "Beer"],
    ["Diapers", "Beer", "Chips"],
    ["Apples", "Bananas"],
    ["Apples", "Bananas", "Milk"],
    ["Bread", "Butter", "Milk", "Eggs"],
    ["Cereal", "Milk"],
    ["Bread", "Butter", "Milk", "Eggs"],
    ["Apples", "Milk"],
    ["Diapers", "Beer", "Bread", "Eggs"],
    ["Yogurt", "Cereal"],
    ["Bread", "Butter", "Bananas", "Milk"],
    ["Milk", "Eggs"],
    ["Apples", "Bananas", "Yogurt"],
    ["Bread", "Butter", "Cereal", "Milk"],
    ["Chips", "Salsa"],
]

In [ ]:
def support(candidate_itemset, transactions):
    """Fraction of transactions that contain every item in candidate_itemset."""
    candidate = frozenset(candidate_itemset)
    count = sum(1 for t in transactions if candidate.issubset(t))
    return count / len(transactions)


def confidence(antecedent, consequent, transactions):
    """Confidence of the rule antecedent -> consequent:
    support(antecedent ∪ consequent) / support(antecedent)."""
    both = set(antecedent) | set(consequent)
    return support(both, transactions) / support(antecedent, transactions)

In [ ]:
def join_step(freq_prev, k):
    """Join frequent (k-1)-itemsets into length-k candidates.
    Itemsets are sorted tuples. Two itemsets are joined only if they share
    the same first k-2 items, which avoids generating duplicates."""
    candidates = []
    for i in range(len(freq_prev)):
        for j in range(i + 1, len(freq_prev)):
            a, b = freq_prev[i], freq_prev[j]
            if a[:k - 2] == b[:k - 2]:
                candidates.append(a + (b[-1],))
    return candidates


def prune_step(candidates, freq_prev):
    """Split candidates into (kept, pruned). A candidate is pruned if any of
    its (k-1)-subsets is not in the frequent (k-1)-itemsets."""
    freq_prev_set = set(freq_prev)
    kept, pruned = [], []
    for c in candidates:
        if all(sub in freq_prev_set for sub in combinations(c, len(c) - 1)):
            kept.append(c)
        else:
            pruned.append(c)
    return kept, pruned

In [ ]:
def apriori(transactions, min_sup):
    """Return (frequent, log).
    frequent: {k: {itemset_tuple: support}}
    log:      {k: info about the join/prune/count steps for level k}"""
    items = sorted({item for t in transactions for item in t})

    # k = 1: count every single item
    L = [(item,) for item in items if support((item,), transactions) >= min_sup]
    frequent = {1: {c: support(c, transactions) for c in L}}
    log = {}

    k = 2
    while L:
        candidates = join_step(L, k)                    # 1. join
        kept, pruned = prune_step(candidates, L)        # 2. prune
        L = [c for c in kept
             if support(c, transactions) >= min_sup]    # 3. count
        log[k] = {"joined": len(candidates), "pruned": pruned,
                  "counted": len(kept), "frequent": len(L)}
        if L:
            frequent[k] = {c: support(c, transactions) for c in L}
        k += 1
    return frequent, log


frequent, log = apriori(transactions, min_sup=0.25)

for k, itemsets in frequent.items():
    print(f"Frequent {k}-itemsets ({len(itemsets)}):")
    for itemset, sup in itemsets.items():
        print(f"   {itemset}  support = {sup:.2f}")

print()
for k, info in log.items():
    print(f"k={k}: {info['joined']} candidates joined, "
          f"{len(info['pruned'])} pruned {info['pruned']}, "
          f"{info['counted']} counted, {info['frequent']} frequent")

## Part 2 Association Rules


## Part 3 Hierarchical Clustering


## Report Answers
